In [32]:
import pandas as pd
import numpy as np
import joblib

In [33]:
rf_cost_model = joblib.load("../models/rf_cost_model.pkl")
xgb_co2_model = joblib.load("../models/xgb_co2_model.pkl")
scaler = joblib.load("../models/feature_scaler.pkl")

In [34]:
df = pd.read_csv("../data/processed/ecopackai_feature_engineered.csv")


In [35]:
features = [
    "material_type",
    "strength",
    "weight_capacity",
    "biodegradability_score",
    "recyclability_percentage",
    "fragility_level",
    "shipping_type"
]

X = df[features]

In [36]:
X = pd.get_dummies(
    X,
    columns=["material_type", "shipping_type"],
    drop_first=True
)

# Add engineered features to match model's feature set
print("Creating engineered features to match RF model...")

# Interaction features
if 'strength' in X.columns and 'weight_capacity' in X.columns:
    X['strength_weight_product'] = X['strength'] * X['weight_capacity']
    strength_max = X['strength'].max()
    weight_max = X['weight_capacity'].max()
    X['strength_weight_normalized'] = (X['strength'] / (strength_max + 1e-6)) * (X['weight_capacity'] / (weight_max + 1e-6))

if 'biodegradability_score' in X.columns and 'recyclability_percentage' in X.columns:
    X['eco_quality_score'] = X['biodegradability_score'] * X['recyclability_percentage']

# Ratio features
if 'strength' in X.columns:
    X['strength_ratio'] = X['strength'] / (X['strength'].max() + 1e-6)

if 'weight_capacity' in X.columns:
    X['weight_capacity_ratio'] = X['weight_capacity'] / (X['weight_capacity'].max() + 1e-6)

# Polynomial features
if 'biodegradability_score' in X.columns:
    X['biodegradability_squared'] = X['biodegradability_score'] ** 2
    X['biodegradability_cubed'] = X['biodegradability_score'] ** 3

if 'recyclability_percentage' in X.columns:
    X['recyclability_squared'] = X['recyclability_percentage'] ** 2

# Material diversity
material_cols = [col for col in X.columns if col.startswith('material_type_')]
if material_cols:
    X['material_diversity'] = X[material_cols].sum(axis=1)

print(f"Features before alignment: {X.shape[1]}")

# Align with model's expected features
X = X.reindex(columns=rf_cost_model.feature_names_in_, fill_value=0)

print(f"Features after alignment: {X.shape[1]}")
print(f"Model expects: {len(rf_cost_model.feature_names_in_)} features")

# Scale numeric columns (only those that were used during training)
num_cols = [
    "strength",
    "weight_capacity",
    "biodegradability_score",
    "recyclability_percentage",
    "fragility_level"
]

# Only scale columns that exist in current data
cols_to_scale = [col for col in num_cols if col in X.columns]
X[cols_to_scale] = scaler.transform(X[cols_to_scale])


Creating engineered features to match RF model...
Features before alignment: 21
Features after alignment: 21
Model expects: 21 features


In [37]:
df["predicted_cost"] = rf_cost_model.predict(X)
df["predicted_co2"]  = xgb_co2_model.predict(X)


In [38]:
def min_max(series):
    if series.max() == series.min():
        return pd.Series(0.0, index=series.index)
    return (series - series.min()) / (series.max() - series.min())

df["cost_norm"] = min_max(df["predicted_cost"])
df["co2_norm"]  = min_max(df["predicted_co2"])


In [39]:
df["ml_rank_score"] = (
    0.5 * (1 - df["cost_norm"]) +     # lower cost → higher score
    0.5 * (1 - df["co2_norm"])        # lower CO₂ → higher score
)


In [40]:
recommendations = (
    df[df["fragility_level"] == 3]
    .groupby("material_type", as_index=False)
    .agg({
        "predicted_cost": "mean",
        "predicted_co2": "mean",
        "ml_rank_score": "mean"
    })
    .sort_values("ml_rank_score", ascending=False)
    .head(5)
)

recommendations


,material_type,predicted_cost,predicted_co2,ml_rank_score
5,paper,0.243727,0.277401,0.898909
0,bagasse,0.250147,0.255615,0.897906
3,jute,0.263430,0.284196,0.846778
1,bamboo,0.296493,0.313364,0.747468
6,plastic,0.320439,0.345686,0.668084


In [41]:
df["ml_rank_score"] = (
    0.4 * (1 - df["cost_norm"]) +
    0.6 * (1 - df["co2_norm"])
)


In [42]:
recommendations = (
    df[df["fragility_level"] == 3]
    .groupby("material_type", as_index=False)
    .agg({
        "predicted_cost": "mean",
        "predicted_co2": "mean",
        "ml_rank_score": "mean"
    })
    .sort_values("ml_rank_score", ascending=False)
    .head(5)
)

recommendations

,material_type,predicted_cost,predicted_co2,ml_rank_score
0,bagasse,0.250147,0.255615,0.889015
5,paper,0.243727,0.277401,0.884014
3,jute,0.263430,0.284196,0.840499
1,bamboo,0.296493,0.313364,0.753281
6,plastic,0.320439,0.345686,0.681164
